In [ ]:
import os
import sys
import pprint
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat, RoundRobinGroupChat
from autogen_agentchat.tools import AgentTool
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import StdioServerParams, mcp_server_tools

# Project import paths
# Get the current directory (which is TFM_BA_GPT/test)
notebook_dir = os.getcwd()

# Define base project directory (one level up from test/)
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))

# Server root: one level up from project_root
code_root = os.path.abspath(os.path.join(project_root, '..'))
server_root = os.path.join(code_root, 'mcp_servers')



# Define paths to each server and Python executable
PyMCP_FS_path = os.path.join(server_root, 'PyMCP-FS')
Mermaid_MCP_path = os.path.join(server_root, 'Mermaid-MCP')
PDF_format_path = os.path.join(server_root, 'Final_deliverable_MCP_server')
tmp_dir = os.path.join(server_root, 'tmp')

# Python executables
PyMCP_FS_python = os.path.join(PyMCP_FS_path, '.venv', 'bin', 'python')
Mermaid_MCP_python = os.path.join(Mermaid_MCP_path, 'venv', 'bin', 'python')
PDF_format_python = os.path.join(PDF_format_path, 'venv', 'bin', 'python')




# Compute absolute paths for lib and src (one level up from test/)
lib_path = os.path.abspath(os.path.join(notebook_dir, '..', 'lib'))
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))

sys.path.append(lib_path)
sys.path.append(src_path)

from model_loader import load_model_client

In [2]:
model_client = load_model_client("azure")

In [3]:
# Helpful sanity prints:
print("raw:", getattr(model_client, "_raw_config", None))
print("create_args:", getattr(model_client, "_create_args", None))

raw: {'model': 'gpt-4o', 'api_key': 'REPLACE_WITH_YOUR_API_KEY', 'azure_endpoint': 'https://224d2-m9kjdmy0-eastus2.openai.azure.com/', 'azure_deployment': 'gpt-4o', 'api_version': '2024-12-01-preview'}
create_args: {'model': 'gpt-4o'}


In [4]:
# Server params
PyMCP_FS = StdioServerParams(
    command=PyMCP_FS_python,
    args=[
        os.path.join(PyMCP_FS_path, 'main.py'),
        '-d', project_root,
        "/tmp/",
        '--log-level', 'DEBUG',
        '--log-file', 'mcp_debug.log'
    ]
)

Mermaid_MCP = StdioServerParams(
    command=Mermaid_MCP_python,
    args=[
        os.path.join(Mermaid_MCP_path, 'Mermaid-MCP.py'),
        '-d', project_root,
        '-d', tmp_dir
    ]
)

PDF_format_mcp_server = StdioServerParams(
    command=PDF_format_python,
    args=[
        os.path.join(PDF_format_path, 'PDF_format_mcp_server.py'),
        '-d', project_root,
        "--llm-config", 
        "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/src/model_config_azure.yaml"
    ]
)


In [5]:
PyMCP_FS_mcp_tools = await mcp_server_tools(PyMCP_FS)
mermaid_mcp_tools = await mcp_server_tools(Mermaid_MCP)
pdf_format_mcp_tools = await mcp_server_tools(PDF_format_mcp_server)

In [6]:

# ---- terminate_agent system message (MOVE + clean + render_mermaid_all + robust PDF) ----
terminate_system_message = r"""
You are the TERMINATE_AGENT. Perform the final export pipeline and then end with **TERMINATE**.

**INPUTS**
  project_root:     /run/media/jun/Jun_6T/Code/TFM_BA_GPT/
  conversation log: project_root/conversation/interview.jsonl
  spec:             project_root/conversation/functional_specification.md
  mermaid:          project_root/conversation/process_diagram.mmd

**GOAL**
  Create: project_root/experience/<SESSION_ID>/
      interview.jsonl
      functional_specification.md
      process_diagram.mmd
      process_diagram.png  
      functional_specification.formatted.md
      deliverable.pdf

**STEPS STRICT ORDER**
  1 Read project_root/conversation/interview.jsonl and extract the first available "session_id".
    If multiple lines: pick the first non-empty "session_id".
    If not found: set SESSION_ID = "sess-unknown".

  2 Create folder project_root/experience/<SESSION_ID>/

  3 MOVE the three source files into that folder using 'move_file':
      project_root/conversation/interview.jsonl             -> project_root/experience/<SESSION_ID>/interview.jsonl
      project_root/conversation/functional_specification.md -> project_root/experience/<SESSION_ID>/functional_specification.md
      project_root/conversation/process_diagram.mmd         -> project_root/experience/<SESSION_ID>/process_diagram.mmd

  4 Clean the Mermaid file **in place** and report issues (severity/code/message/recommendation):
      clean_mermaid(
        input_path="project_root/experience/<SESSION_ID>/process_diagram.mmd",
        output_path="project_root/experience/<SESSION_ID>/process_diagram.mmd",
        overwrite=True
      )

  5 Render Mermaid **PNG**:
      render_mermaid_png(
        input_path="project_root/experience/<SESSION_ID>/process_diagram.mmd",
        output_path="project_root/experience/<SESSION_ID>/process_diagram.png",
        theme="forest"
      )

  6 Run **format_and_export** ONLY:
      format_and_export(
        session_dir="project_root/experience/<SESSION_ID>",
        md="functional_specification.md",
        diagram="process_diagram.png",
        md_out="functional_specification.formatted.md",
        pdf_out="deliverable.pdf",
        engine="xelatex",
        toc=True, toc_depth=2, margin="1in",
        mainfont="DejaVu Serif", monofont="DejaVu Sans Mono"
      )

  7 Confirm project_root/experience/<SESSION_ID>/deliverable.pdf exists.

  8 Save experience metadata (local Mem0):
      save_experience_to_mem0(
        session_dir="project_root/experience/<SESSION_ID>",
        session_id="<SESSION_ID>"

  9 Output exactly: **TERMINATE**

**RULES**
  Be explicit about each steps success or failure.
  If a tool call fails, retry up to 1 times with the same arguments.
"""


In [7]:
termination = TextMentionTermination("TERMINATE")

terminate_agent = AssistantAgent(
    name="terminate_agent",
    model_client=model_client,
    system_message=terminate_system_message,
    tools=(PyMCP_FS_mcp_tools + mermaid_mcp_tools + pdf_format_mcp_tools),
    reflect_on_tool_use=False,
)
team = RoundRobinGroupChat(
    [terminate_agent],
    termination_condition=termination,
    )
await Console(team.run_stream(task="Execute the termination pipeline"))

---------- TextMessage (user) ----------
Execute the termination pipeline
---------- ToolCallRequestEvent (terminate_agent) ----------
[FunctionCall(id='call_cjTsWJ5E03rrIUAbe0adT6h8', arguments='{"path":"/run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/interview.jsonl"}', name='read_file')]


/run/media/jun/Jun_6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/agents/_assistant_agent.py:1107: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


---------- ToolCallExecutionEvent (terminate_agent) ----------
[FunctionExecutionResult(content='[{"type": "text", "text": "{\\"_meta\\": {\\"session_id\\": \\"sess-20231011162620\\", \\"ts\\": \\"2025-08-09T20:35:33.157973Z\\", \\"n_records\\": 2}}\\n{\\"speaker\\": \\"interviewer\\", \\"message\\": \\"QUESTION: Can you briefly describe the new product and its primary purpose?\\", \\"subject\\": \\"Product Overview\\", \\"q_index\\": 1, \\"timestamp\\": \\"2025-08-09T20:35:33.157947Z\\"}\\n{\\"speaker\\": \\"user\\", \\"message\\": \\"The new product is an AI dialer that calls customers to offer travel packages. If the customer accepts, the call transfers to a human. If declined, the AI attempts to convince the customer up to three additional times before ending the call\\", \\"subject\\": \\"Product Overview\\", \\"q_index\\": 1, \\"timestamp\\": \\"2025-08-09T20:35:33.157947Z\\"}\\n{\\"_meta\\": {\\"session_id\\": \\"sess-20231011162620\\", \\"ts\\": \\"2025-08-09T20:36:12.337556Z\\

/run/media/jun/Jun_6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/agents/_assistant_agent.py:1107: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


---------- ToolCallExecutionEvent (terminate_agent) ----------
[FunctionExecutionResult(content='[{"type": "text", "text": "Successfully created directory /run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620/", "annotations": null}]', name='create_directory', call_id='call_xkDGkK7imtYy3U0wLNwRdfYk', is_error=False)]
---------- ToolCallSummaryMessage (terminate_agent) ----------
[{"type": "text", "text": "Successfully created directory /run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620/", "annotations": null}]
---------- ToolCallRequestEvent (terminate_agent) ----------
[FunctionCall(id='call_e81iWnxZuIWSoo7hXf1hgWpR', arguments='{"source": "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/interview.jsonl", "destination": "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620/interview.jsonl"}', name='move_file'), FunctionCall(id='call_dMQtKDc7nN36Cyh4ePHKKsti', arguments='{"source": "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/f

/run/media/jun/Jun_6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/agents/_assistant_agent.py:1107: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


... (11 outputs left out)

/run/media/jun/Jun_6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/agents/_assistant_agent.py:1107: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


---------- ToolCallExecutionEvent (terminate_agent) ----------
[FunctionExecutionResult(content='[{"type": "text", "text": "{\\"ok\\": true, \\"db_root\\": \\"/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.mem0_local\\", \\"collection\\": \\"tfm_ba_gpt_experiences\\", \\"session_id\\": \\"sess-20231011162620\\", \\"files_indexed\\": 6}", "annotations": null}]', name='save_experience_to_mem0', call_id='call_gcKNdFA1Ffk9dLrdxkSZmW2h', is_error=False)]
---------- ToolCallSummaryMessage (terminate_agent) ----------
[{"type": "text", "text": "{\"ok\": true, \"db_root\": \"/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.mem0_local\", \"collection\": \"tfm_ba_gpt_experiences\", \"session_id\": \"sess-20231011162620\", \"files_indexed\": 6}", "annotations": null}]
---------- TextMessage (terminate_agent) ----------
**TERMINATE**


/run/media/jun/Jun_6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/agents/_assistant_agent.py:1107: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


TaskResult(messages=[TextMessage(id='76559528-2514-4414-b04d-809915eefcf0', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 8, 22, 15, 25, 34, 256435, tzinfo=datetime.timezone.utc), content='Execute the termination pipeline', type='TextMessage'), ToolCallRequestEvent(id='26eda647-6a14-4219-9101-2d47879f8407', source='terminate_agent', models_usage=RequestUsage(prompt_tokens=3124, completion_tokens=39), metadata={}, created_at=datetime.datetime(2025, 8, 22, 15, 25, 35, 654875, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_cjTsWJ5E03rrIUAbe0adT6h8', arguments='{"path":"/run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/interview.jsonl"}', name='read_file')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='2093791b-56f8-49a4-b056-bebe5357c28e', source='terminate_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 8, 22, 15, 25, 35, 907983, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(conte